# 01 · Data understanding (Unit I)

Python essentials, NumPy, pandas, data types, missing values, duplicates, encoding, scaling and the train/validation/test split.

Data: UCI Bike Sharing `hour.csv` (official source, CC BY 4.0).

In [1]:
# Make the project folder importable (notebooks live in notebooks/)
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import Image, display
from config import project_config as cfg
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Load and validate the real dataset
`load_hour_data()` stops with an error if the file does not match the official UCI format.

In [2]:
from src.data_loader import load_hour_data, summarise_data
df = load_hour_data()
print(df.shape)
df.head()

(17379, 18)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,datetime
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16,2011-01-01 00:00:00
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40,2011-01-01 01:00:00
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32,2011-01-01 02:00:00
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13,2011-01-01 03:00:00
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1,2011-01-01 04:00:00


## 2. Data types and statistical summary

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 18 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   instant     17379 non-null  int64         
 1   dteday      17379 non-null  datetime64[us]
 2   season      17379 non-null  int64         
 3   yr          17379 non-null  int64         
 4   mnth        17379 non-null  int64         
 5   hr          17379 non-null  int64         
 6   holiday     17379 non-null  int64         
 7   weekday     17379 non-null  int64         
 8   workingday  17379 non-null  int64         
 9   weathersit  17379 non-null  int64         
 10  temp        17379 non-null  float64       
 11  atemp       17379 non-null  float64       
 12  hum         17379 non-null  float64       
 13  windspeed   17379 non-null  float64       
 14  casual      17379 non-null  int64         
 15  registered  17379 non-null  int64         
 16  cnt         17379 non-null  int64

In [4]:
df[cfg.HOUR_COLUMNS].describe().T.round(3)

,count,mean,min,25%,50%,75%,max,std
instant,17379.0,8690.0,1.0,4345.5,8690.0,13034.5,17379.0,5017.0295
dteday,17379,2012-01-02 04:08:34.552045,2011-01-01 00:00:00,2011-07-04 00:00:00,2012-01-02 00:00:00,2012-07-02 00:00:00,2012-12-31 00:00:00,NaN
season,17379.0,2.50164,1.0,2.0,3.0,3.0,4.0,1.106918
yr,17379.0,0.502561,0.0,0.0,1.0,1.0,1.0,0.500008
mnth,17379.0,6.537775,1.0,4.0,7.0,10.0,12.0,3.438776
hr,17379.0,11.546752,0.0,6.0,12.0,18.0,23.0,6.914405
holiday,17379.0,0.02877,0.0,0.0,0.0,0.0,1.0,0.167165
weekday,17379.0,3.003683,0.0,1.0,3.0,5.0,6.0,2.005771
workingday,17379.0,0.682721,0.0,0.0,1.0,1.0,1.0,0.465431
weathersit,17379.0,1.425283,1.0,1.0,1.0,2.0,4.0,0.639357


## 3. NumPy: arrays and vectorised operations
UCI stores temperature on a 0–1 scale. With NumPy we convert the whole column to °C in one operation (no loop).

In [5]:
cnt = df['cnt'].to_numpy()          # pandas column -> NumPy array
print(type(cnt), cnt.shape, cnt.dtype)
print('mean', cnt.mean().round(2), '| std', cnt.std().round(2),
      '| 25/50/75th percentiles', np.percentile(cnt, [25, 50, 75]))
temp_c = df['temp'].to_numpy() * (cfg.TEMP_MAX_C - cfg.TEMP_MIN_C) + cfg.TEMP_MIN_C
print('temperature range in °C:', temp_c.min().round(1), 'to', temp_c.max().round(1))
print('hours with more than 500 rentals:', np.sum(cnt > 500))

<class 'numpy.ndarray'> (17379,) int64
mean 189.46 | std 181.38 | 25/50/75th percentiles [ 40. 142. 281.]
temperature range in °C: -7.1 to 39.0
hours with more than 500 rentals: 1279


## 4. Missing values, duplicates and missing hours

In [6]:
print('Missing values per column:')
print(df[cfg.HOUR_COLUMNS].isna().sum().to_string())
summary = summarise_data(df)
{k: summary[k] for k in ['rows', 'missing_values_total', 'duplicate_rows', 'duplicate_timestamps', 'hours_in_date_range', 'missing_hours']}

Missing values per column:
instant       0
dteday        0
season        0
yr            0
mnth          0
hr            0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
casual        0
registered    0
cnt           0


{'rows': 17379,
 'missing_values_total': 0,
 'duplicate_rows': 0,
 'duplicate_timestamps': 0,
 'hours_in_date_range': 17544,
 'missing_hours': 165}

**Finding:** the file has **no missing cell values** and **no duplicate rows**, so no imputation or de-duplication is needed. We do **not** invent values.

Separately, 165 hours have **no row at all** (for example the Hurricane Sandy shutdown). These are not filled in, because inventing rental counts would be fabricating data.

**Strategy if missing values appeared** (for example in user input): reject invalid input (`src/predict.py`), or put a `SimpleImputer` *inside* the pipeline so it is fitted on training data only. The check below confirms the imputer would change nothing here.

In [7]:
from sklearn.impute import SimpleImputer
num = df[['temp', 'hum', 'windspeed']]
imputed = SimpleImputer(strategy='median').fit_transform(num)
print('Values changed by imputation:', int((imputed != num.to_numpy()).sum()))

Values changed by imputation: 0


## 5. Categorical encoding
`season`, `weathersit`, `hr` etc. are **codes**, not quantities (season 4 is not 'twice' season 2). Linear models therefore get **one-hot** columns.

In [8]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder(sparse_output=False).fit(df[['season']])
pd.DataFrame(enc.transform(df[['season']].head(3)), columns=enc.get_feature_names_out())

,season_1,season_2,season_3,season_4
0,1.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0


## 6. Chronological train / validation / test split
Random shuffling would let the model see the future (demand grew ~63% from 2011 to 2012), so we split by date.

In [9]:
from src.feature_engineering import add_features
from src.preprocessing import chronological_split, split_summary
train, val, test = chronological_split(add_features(df))
split_summary(train, val, test)

,split,rows,percent,start,end
0,train,13003,74.8,2011-01-01,2012-06-30 23:00:00
1,validation,2208,12.7,2012-07-01,2012-09-30 23:00:00
2,test,2168,12.5,2012-10-01,2012-12-31 23:00:00


## 7. Feature scaling (fitted on training data only)
`StandardScaler` learns mean and standard deviation **from the training period**, then applies them unchanged to validation and test.

In [10]:
from sklearn.preprocessing import StandardScaler
cols = ['temp', 'hum', 'windspeed']
scaler = StandardScaler().fit(train[cols])
print('means learned from train:', scaler.mean_.round(3))
print('validation after scaling -> mean', scaler.transform(val[cols]).mean(axis=0).round(2),
      '(not exactly 0, because the scaler never saw validation data)')

means learned from train: [0.482 0.624 0.196]
validation after scaling -> mean [ 1.12  0.05 -0.25] (not exactly 0, because the scaler never saw validation data)
